# MAIN CODE

In [ ]:
import os
import re
import time
import hashlib
import requests
import pandas as pd
import xml.etree.ElementTree as ET
import shutil
from io import BytesIO
from PIL import Image
from bs4 import BeautifulSoup
from datetime import datetime
from urllib.parse import urlparse, urljoin

In [ ]:
print("Working directory:")
print(os.getcwd())

print("\nFull image folder path:")
print(os.path.abspath("spiegel_images"))

In [ ]:
def count_articles_only(sitemap_index_url, years, allowed_sections):
    import xml.etree.ElementTree as ET
    from datetime import datetime
    import requests
    import time

    namespace = {"ns": "http://www.sitemaps.org/schemas/sitemap/0.9"}

    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0"
    })

    def get_xml_root(url):
        response = session.get(url, timeout=30)
        response.raise_for_status()
        return ET.fromstring(response.content)

    index_root = get_xml_root(sitemap_index_url)

    sitemap_urls = [
        elem.text
        for elem in index_root.findall("ns:sitemap/ns:loc", namespace)
        if elem.text and any(year in elem.text for year in years)
    ]

    total = 0
    per_section = {}
    per_month = {}

    for sitemap_url in sitemap_urls:
        root = get_xml_root(sitemap_url)

        for url_elem in root.findall("ns:url", namespace):
            loc = url_elem.find("ns:loc", namespace)
            lastmod = url_elem.find("ns:lastmod", namespace)

            if loc is None or lastmod is None:
                continue

            url = loc.text
            date = lastmod.text

            if not url or not date:
                continue

            try:
                dt = datetime.fromisoformat(date.replace("Z", "+00:00"))
            except:
                continue

            if str(dt.year) not in years:
                continue

            parts = url.split("/")
            section = parts[3] if len(parts) > 3 else "unknown"

            if section not in allowed_sections:
                continue

            year_month = f"{dt.year}-{dt.month:02d}"

            total += 1

            per_section[section] = per_section.get(section, 0) + 1
            per_month[year_month] = per_month.get(year_month, 0) + 1

        time.sleep(0.05)

    print("Total articles:", total)

    print("\nBy section:")
    for k, v in sorted(per_section.items(), key=lambda x: -x[1]):
        print(k, v)

    print("\nBy month:")
    for k, v in sorted(per_month.items()):
        print(k, v)

In [ ]:
count_articles_only(
    sitemap_index_url="https://www.origo.hu/sitemapindex.xml",
    years=["2023", "2024", "2025", "2026"],
    allowed_sections={"nagyvilag", "belpol", "kulpol", "gazdasag"}
)

## Test 100

In [ ]:
#settings
sitemap_index_url = "https://www.spiegel.de/sitemap.xml"
years = ["2023", "2024", "2025", "2026"]
allowed_sections = {"ausland", "wirtschaft", "politik"}

output_base_folder = "spiegel_test_100"
images_folder = os.path.join(output_base_folder, "images")
metadata_csv_path = os.path.join(output_base_folder, "spiegel_test_1000_metadata.csv")

target_image_count = 1000
min_size = 300

request_timeout_xml = 30
request_timeout_html = 30
request_timeout_image = 20
sleep_between_requests = 0.2

os.makedirs(output_base_folder, exist_ok=True)
os.makedirs(images_folder, exist_ok=True)

session = requests.Session()
session.headers.update(
    {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0 Safari/537.36"
    }
)

namespace = {"ns": "http://www.sitemaps.org/schemas/sitemap/0.9"}

In [ ]:

#helpers
def get_xml_root(url):
    response = session.get(url, timeout=request_timeout_xml)
    response.raise_for_status()
    return ET.fromstring(response.content)


def clean_text(text):
    if text is None:
        return None
    text = re.sub(r"\s+", " ", text).strip()
    return text if text else None


def sanitize_filename_part(text, max_length=80):
    if text is None:
        return "unknown"
    text = re.sub(r"[^\w\-]+", "_", str(text), flags=re.UNICODE)
    text = re.sub(r"_+", "_", text).strip("_")
    if not text:
        text = "unknown"
    return text[:max_length]


def get_file_extension_from_url(image_url):
    parsed = urlparse(image_url)
    path = parsed.path.lower()

    if path.endswith(".png"):
        return ".png"
    if path.endswith(".webp"):
        return ".webp"
    if path.endswith(".jpeg"):
        return ".jpg"
    if path.endswith(".jpg"):
        return ".jpg"

    return ".jpg"


def build_image_filename(image_url, year_month, section, image_index):
    url_hash = hashlib.md5(image_url.encode("utf-8")).hexdigest()[:16]
    extension = get_file_extension_from_url(image_url)
    safe_year_month = sanitize_filename_part(year_month, max_length=20)
    safe_section = sanitize_filename_part(section, max_length=40)
    return f"{safe_year_month}_{safe_section}_{image_index:04d}_{url_hash}{extension}"


def download_image(image_url, save_path, min_size=300):
    try:
        response = session.get(image_url, timeout=request_timeout_image, stream=True)
        response.raise_for_status()

        content_type = response.headers.get("Content-Type", "")
        if "image" not in content_type.lower():
            return None, None, None

        image = Image.open(BytesIO(response.content))

        if image.width < min_size or image.height < min_size:
            return None, None, None

        image_rgb = image.convert("RGB")
        final_save_path = os.path.splitext(save_path)[0] + ".jpg"
        image_rgb.save(final_save_path, "JPEG", quality=95)

        return final_save_path, image.width, image.height

    except Exception:
        return None, None, None


#sitemap collection
def collect_article_urls(
    sitemap_index_url,
    years,
    allowed_sections
):
    index_root = get_xml_root(sitemap_index_url)

    all_sitemap_urls = [
        elem.text
        for elem in index_root.findall("ns:sitemap/ns:loc", namespace)
        if elem.text
    ]

    sitemap_filtered = [
        url for url in all_sitemap_urls
        if any(year in url for year in years)
    ]

    print(f"Found {len(sitemap_filtered)} sitemap files for {years[0]}-{years[-1]}.")

    articles_data = []

    for sitemap_url in sitemap_filtered:
        try:
            root = get_xml_root(sitemap_url)

            for url_elem in root.findall("ns:url", namespace):
                loc_elem = url_elem.find("ns:loc", namespace)
                lastmod_elem = url_elem.find("ns:lastmod", namespace)

                if loc_elem is None or lastmod_elem is None:
                    continue

                loc = loc_elem.text
                lastmod = lastmod_elem.text

                if not loc or not lastmod:
                    continue

                try:
                    lastmod_date = datetime.fromisoformat(lastmod.replace("Z", "+00:00"))
                except Exception:
                    continue

                if str(lastmod_date.year) not in years:
                    continue

                parts = loc.split("/")
                section = parts[3] if len(parts) > 3 else "unknown"

                if section not in allowed_sections:
                    continue

                year_month = f"{lastmod_date.year}-{lastmod_date.month:02d}"

                articles_data.append(
                    {
                        "article_url": loc,
                        "year_month": year_month,
                        "section": section,
                        "lastmod": lastmod
                    }
                )


        except Exception as e:
            print(f"Error processing sitemap {sitemap_url}: {e}")

        time.sleep(sleep_between_requests)

    articles_df = pd.DataFrame(articles_data)
    articles_df = articles_df.drop_duplicates(subset=["article_url"]).reset_index(drop=True)

    print(f"Collected {len(articles_df)} article URLs after section filtering.")
    return articles_df


#article parsing
def extract_images_from_article(article_url, year_month, section, lastmod):
    results = []

    try:
        response = session.get(article_url, timeout=request_timeout_html)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        headline_tag = soup.find("h1")
        headline = clean_text(headline_tag.get_text(" ", strip=True)) if headline_tag else None

        figures = soup.find_all("figure")
        image_counter_in_article = 0

        for fig in figures:
            img = fig.find("img")
            if img is None:
                continue

            image_url = img.get("src") or img.get("data-src")

            if not image_url:
                srcset = img.get("srcset")
                if srcset:
                    first_src = srcset.split(",")[0].strip()
                    image_url = first_src.split(" ")[0]

            if not image_url:
                continue

            if image_url.startswith("data:image"):
                continue

            image_url = urljoin(article_url, image_url)

            caption_tag = fig.find("figcaption")
            caption_text = clean_text(caption_tag.get_text(" ", strip=True)) if caption_tag else None
            alt_text = clean_text(img.get("alt"))
            title_text = clean_text(img.get("title"))

            image_counter_in_article += 1

            results.append(
                {
                    "article_url": article_url,
                    "headline": headline,
                    "year_month": year_month,
                    "section": section,
                    "lastmod": lastmod,
                    "image_url": image_url,
                    "caption": caption_text,
                    "alt_text": alt_text,
                    "title_text": title_text,
                    "image_index_in_article": image_counter_in_article
                }
            )

    except Exception as e:
        print(f"Error processing article {article_url}: {e}")

    return results


In [ ]:
#main pipeline
articles_df = collect_article_urls(
    sitemap_index_url=sitemap_index_url,
    years=years,
    allowed_sections=allowed_sections
)

saved_rows = []
seen_image_urls = set()
scanned_articles = 0

for _, row in articles_df.iterrows():
    if len(saved_rows) >= target_image_count:
        break

    article_url = row["article_url"]
    year_month = row["year_month"]
    section = row["section"]
    lastmod = row["lastmod"]

    scanned_articles += 1
    print(
        f"Scanning article {scanned_articles}/{len(articles_df)} | "
        f"saved {len(saved_rows)}/{target_image_count}"
    )

    image_candidates = extract_images_from_article(
        article_url=article_url,
        year_month=year_month,
        section=section,
        lastmod=lastmod
    )

    for image_data in image_candidates:
        if len(saved_rows) >= target_image_count:
            break

        image_url = image_data["image_url"]

        if not image_url or image_url in seen_image_urls:
            continue

        filename = build_image_filename(
            image_url=image_url,
            year_month=image_data["year_month"],
            section=image_data["section"],
            image_index=len(saved_rows) + 1
        )

        save_path = os.path.join(images_folder, filename)

        local_path, width, height = download_image(
            image_url=image_url,
            save_path=save_path,
            min_size=min_size
        )

        if local_path is None:
            continue

        seen_image_urls.add(image_url)

        saved_rows.append(
            {
                "image_id": len(saved_rows) + 1,
                "headline": image_data["headline"],
                "article_url": image_data["article_url"],
                "year_month": image_data["year_month"],
                "section": image_data["section"],
                "lastmod": image_data["lastmod"],
                "image_url": image_data["image_url"],
                "caption": image_data["caption"],
                "alt_text": image_data["alt_text"],
                "title_text": image_data["title_text"],
                "image_index_in_article": image_data["image_index_in_article"],
                "local_image_path": local_path
            }
        )

        print(f"Saved {len(saved_rows)}/{target_image_count}: {local_path}")

    time.sleep(sleep_between_requests)

images_df = pd.DataFrame(saved_rows)
images_df.to_csv(metadata_csv_path, index=False, encoding="utf-8-sig")

In [ ]:
images_df = pd.read_csv("spiegel_test_100_metadata.csv")

In [ ]:
images_df

### Only Ukr

In [ ]:
import re
import pandas as pd

anchor_pattern = re.compile(
    r"\b("
    r"ukrain\w*|"
    r"ukraine|"
    r"ukrainisch\w*|"
    r"russ\w*|"
    r"russland\w*|"
    r"russisch\w*|"
    r"moskau\w*|"
    r"kreml\w*|"
    r"kiew\w*|"
    r"kyjiw\w*|"
    r"putin\w*|"
    r"selensk\w*|"
    r"zelensk\w*|"
    r"lawrow\w*|"
    r"prigoschin\w*|"
    r"wagner\w*|"
    r"medwedew\w*|"
    r"donbas\w*|"
    r"donezk\w*|"
    r"donez\w*|"
    r"luhansk\w*|"
    r"krim\w*|"
    r"mariupol\w*|"
    r"charkiw\w*|"
    r"cherson\w*|"
    r"odesa\w*|"
    r"odessa\w*|"
    r"bachmut\w*|"
    r"avdiiw\w*|"
    r"sumy\w*|"
    r"dnipro\w*|"
    r"saporisch\w*"
    r")\b",
    flags=re.IGNORECASE
)

war_pattern = re.compile(
    r"\b("
    r"krieg\w*|"
    r"invasion\w*|"
    r"angriff\w*|"
    r"offensive\w*|"
    r"gegenoffensive\w*|"
    r"front\w*|"
    r"kampf\w*|"
    r"gefecht\w*|"
    r"beschuss\w*|"
    r"bomb\w*|"
    r"raket\w*|"
    r"drohnen\w*|"
    r"luftangriff\w*|"
    r"milit\w*|"
    r"soldat\w*|"
    r"trupp\w*|"
    r"armee\w*|"
    r"verteidig\w*|"
    r"besatz\w*|"
    r"annekt\w*|"
    r"sanktion\w*|"
    r"waffen\w*|"
    r"munition\w*|"
    r"panzer\w*|"
    r"artillerie\w*|"
    r"flugabwehr\w*|"
    r"zivilist\w*|"
    r"flucht\w*|"
    r"fluecht\w*|"
    r"flücht\w*|"
    r"tote\w*|"
    r"verletz\w*|"
    r"zerstoer\w*|"
    r"zerstör\w*|"
    r"ruine\w*|"
    r"friedens\w*|"
    r"verhandlung\w*|"
    r"atom\w*|"
    r"nuklear\w*"
    r")\b",
    flags=re.IGNORECASE
)

text_columns = ["headline", "caption", "alt_text", "title_text"]

for col in text_columns:
    if col not in images_df.columns:
        images_df[col] = ""

images_df["combined_text"] = (
    images_df["headline"].fillna("") + " "
    + images_df["caption"].fillna("") + " "
    + images_df["alt_text"].fillna("") + " "
    + images_df["title_text"].fillna("")
).str.lower()

images_df["anchor_matches"] = images_df["combined_text"].apply(
    lambda text: sorted(set(anchor_pattern.findall(text)))
)

images_df["war_matches"] = images_df["combined_text"].apply(
    lambda text: sorted(set(war_pattern.findall(text)))
)

images_df["anchor_match"] = images_df["anchor_matches"].apply(lambda x: len(x) > 0)
images_df["war_match"] = images_df["war_matches"].apply(lambda x: len(x) > 0)

images_df["ru_ua_war_match"] = images_df["anchor_match"] & images_df["war_match"]

related_df = images_df[images_df["ru_ua_war_match"]].copy()

print("Total image rows:", len(images_df))
print("Rows with anchor match:", images_df["anchor_match"].sum())
print("Rows with war match:", images_df["war_match"].sum())
print("Rows with RU/UA war match:", len(related_df))

related_df[
    [
        "image_id",
        "headline",
        "caption",
        "anchor_matches",
        "war_matches",
        "section",
        "year_month"
    ]
].head(20)

In [ ]:
for i, caption in enumerate(related_df["caption"].fillna(""), start=1):
    print(f"\n{i}. {caption}")

In [ ]:
#new folder inside images
filtered_folder = os.path.join(images_folder, "ukraine_related")
os.makedirs(filtered_folder, exist_ok=True)

copied = 0

for _, row in ukraine_related_df.iterrows():
    source_path = row["local_image_path"]

    if not os.path.exists(source_path):
        continue

    filename = os.path.basename(source_path)
    target_path = os.path.join(filtered_folder, filename)

    #avoid overwriting duplicates
    if not os.path.exists(target_path):
        shutil.copy2(source_path, target_path)
        copied += 1

print(f"Copied {copied} images to {filtered_folder}")

## ORIGO

In [ ]:
#DIAGNOSTICS

#settings
sitemap_index_url = "https://www.origo.hu/sitemapindex.xml"
max_sitemaps_to_check = 5
max_article_urls_per_sitemap = 5
request_timeout = 30

session = requests.Session()
session.headers.update(
    {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0 Safari/537.36"
    }
)

namespace = {"ns": "http://www.sitemaps.org/schemas/sitemap/0.9"}


#helpers
def clean_text(text):
    if text is None:
        return None
    text = re.sub(r"\s+", " ", text).strip()
    return text if text else None


def get_xml_root(url):
    response = session.get(url, timeout=request_timeout)
    response.raise_for_status()
    return ET.fromstring(response.content)


def safe_find_text(parent, xpath, namespace):
    elem = parent.find(xpath, namespace)
    if elem is None:
        return None
    return elem.text


def guess_section_from_url(article_url):
    parsed = urlparse(article_url)
    path_parts = [part for part in parsed.path.split("/") if part]

    if len(path_parts) == 0:
        return None

    #usually the first path segment is the section candidate
    return path_parts[0]


def extract_article_diagnostics(article_url):
    result = {
        "article_url": article_url,
        "headline_h1": None,
        "headline_og_title": None,
        "headline_title_tag": None,
        "og_image": None,
        "figure_img_src": None,
        "figure_img_alt": None,
        "figcaption": None,
        "section_guess_from_url": guess_section_from_url(article_url),
    }

    try:
        response = session.get(article_url, timeout=request_timeout)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        #headline candidates
        h1_tag = soup.find("h1")
        result["headline_h1"] = clean_text(h1_tag.get_text(" ", strip=True)) if h1_tag else None

        og_title_tag = soup.find("meta", attrs={"property": "og:title"})
        result["headline_og_title"] = clean_text(og_title_tag.get("content")) if og_title_tag else None

        title_tag = soup.find("title")
        result["headline_title_tag"] = clean_text(title_tag.get_text(" ", strip=True)) if title_tag else None

        #image candidates
        og_image_tag = soup.find("meta", attrs={"property": "og:image"})
        result["og_image"] = clean_text(og_image_tag.get("content")) if og_image_tag else None

        first_figure = soup.find("figure")
        if first_figure is not None:
            img_tag = first_figure.find("img")
            if img_tag is not None:
                result["figure_img_src"] = clean_text(
                    img_tag.get("src") or img_tag.get("data-src") or img_tag.get("srcset")
                )
                result["figure_img_alt"] = clean_text(img_tag.get("alt"))

            figcaption_tag = first_figure.find("figcaption")
            result["figcaption"] = clean_text(figcaption_tag.get_text(" ", strip=True)) if figcaption_tag else None

        return result

    except Exception as e:
        result["error"] = str(e)
        return result


#step 1: open sitemap index
print("#step 1: loading sitemap index")
index_root = get_xml_root(sitemap_index_url)

sitemap_entries = index_root.findall("ns:sitemap", namespace)
print(f"Total sitemap entries found in sitemap index: {len(sitemap_entries)}")
print()

#step 2: inspect first sitemap urls
print("#step 2: first sitemap urls")
selected_sitemap_urls = []

for i, sitemap_elem in enumerate(sitemap_entries[:max_sitemaps_to_check], start=1):
    sitemap_url = safe_find_text(sitemap_elem, "ns:loc", namespace)
    lastmod = safe_find_text(sitemap_elem, "ns:lastmod", namespace)

    selected_sitemap_urls.append(sitemap_url)

    print(f"{i}. sitemap_url: {sitemap_url}")
    print(f"   lastmod: {lastmod}")

print()

#step 3: inspect article urls inside first few sitemaps
print("#step 3: inspect article urls inside selected sitemaps")

all_sample_article_urls = []

for sitemap_url in selected_sitemap_urls:
    print()
    print(f"Reading sitemap: {sitemap_url}")

    try:
        sitemap_root = get_xml_root(sitemap_url)
        url_entries = sitemap_root.findall("ns:url", namespace)
        print(f"URL entries in this sitemap: {len(url_entries)}")

        sample_urls = []

        for url_elem in url_entries[:max_article_urls_per_sitemap]:
            article_url = safe_find_text(url_elem, "ns:loc", namespace)
            lastmod = safe_find_text(url_elem, "ns:lastmod", namespace)

            if article_url:
                sample_urls.append(article_url)
                all_sample_article_urls.append(article_url)

                section_guess = guess_section_from_url(article_url)

                print(f"  article_url: {article_url}")
                print(f"  lastmod: {lastmod}")
                print(f"  section_guess_from_url: {section_guess}")
                print()

    except Exception as e:
        print(f"  error reading sitemap: {e}")

print()

#step 4: inspect html structure for a few article pages
print("#step 4: inspect article html structure")

unique_sample_article_urls = []
seen_urls = set()

for url in all_sample_article_urls:
    if url not in seen_urls:
        unique_sample_article_urls.append(url)
        seen_urls.add(url)

for article_url in unique_sample_article_urls[:5]:
    print()
    print("=" * 100)
    diagnostics = extract_article_diagnostics(article_url)

    print(f"article_url: {diagnostics['article_url']}")
    print(f"section_guess_from_url: {diagnostics.get('section_guess_from_url')}")
    print()
    print(f"headline_h1: {diagnostics.get('headline_h1')}")
    print(f"headline_og_title: {diagnostics.get('headline_og_title')}")
    print(f"headline_title_tag: {diagnostics.get('headline_title_tag')}")
    print()
    print(f"og_image: {diagnostics.get('og_image')}")
    print(f"figure_img_src: {diagnostics.get('figure_img_src')}")
    print(f"figure_img_alt: {diagnostics.get('figure_img_alt')}")
    print(f"figcaption: {diagnostics.get('figcaption')}")

    if "error" in diagnostics:
        print()
        print(f"error: {diagnostics['error']}")

In [ ]:
from urllib.parse import urlparse, urljoin
from datetime import datetime
from bs4 import BeautifulSoup
import pandas as pd
import re
import os
import time

#settings
sitemap_index_url = "https://www.origo.hu/sitemapindex.xml"
start_year = 2023
allowed_sections = {"nagyvilag", "belpol", "kulpol", "gazdasag"}

output_base_folder = "origo_test_100"
images_folder = os.path.join(output_base_folder, "images")
metadata_csv_path = os.path.join(output_base_folder, "origo_test_100_metadata.csv")

target_image_count = 100
max_articles_to_scan = 500
min_size = 300
request_timeout_html = 30

os.makedirs(output_base_folder, exist_ok=True)
os.makedirs(images_folder, exist_ok=True)

#origo-specific helpers
def parse_iso_datetime(value):
    if value is None:
        return None
    try:
        return datetime.fromisoformat(value.replace("Z", "+00:00"))
    except Exception:
        return None


def guess_section_from_url(article_url):
    parsed = urlparse(article_url)
    path_parts = [part for part in parsed.path.split("/") if part]
    if len(path_parts) == 0:
        return None
    return path_parts[0]


def is_probable_article_url(article_url):
    if not article_url:
        return False

    parsed = urlparse(article_url)
    path_parts = [part for part in parsed.path.split("/") if part]

    if article_url.rstrip("/") == "https://www.origo.hu":
        return False

    if len(path_parts) < 4:
        return False

    if not path_parts[1].isdigit() or len(path_parts[1]) != 4:
        return False

    if not path_parts[2].isdigit() or len(path_parts[2]) != 2:
        return False

    return True


def should_keep_sitemap(sitemap_url, lastmod_text, start_year):
    if not sitemap_url:
        return False

    filename = sitemap_url.split("/")[-1].lower()

    if "home_page" in filename:
        return False

    if re.match(r"189912_sitemap\d*\.xml$", filename):
        return False

    dt = parse_iso_datetime(lastmod_text)
    if dt is not None and dt.year >= start_year:
        return True

    year_month_match = re.match(r"(\d{4})(\d{2})_sitemap\d*\.xml$", filename)
    if year_month_match:
        year = int(year_month_match.group(1))
        return year >= start_year

    if "videos_sitemap" in filename:
        return True

    return False


def extract_first_image_candidate(soup, article_url):
    first_figure = soup.find("figure")
    image_url = None
    alt_text = None
    caption_text = None

    if first_figure is not None:
        img_tag = first_figure.find("img")
        if img_tag is not None:
            image_url = img_tag.get("src") or img_tag.get("data-src")

            if not image_url:
                srcset = img_tag.get("srcset")
                if srcset:
                    first_src = srcset.split(",")[0].strip()
                    image_url = first_src.split(" ")[0]

            alt_text = clean_text(img_tag.get("alt"))

        figcaption_tag = first_figure.find("figcaption")
        caption_text = clean_text(figcaption_tag.get_text(" ", strip=True)) if figcaption_tag else None

    if not image_url:
        og_image_tag = soup.find("meta", attrs={"property": "og:image"})
        if og_image_tag is not None:
            image_url = og_image_tag.get("content")

    if image_url:
        image_url = urljoin(article_url, image_url)

    return image_url, alt_text, caption_text


def collect_origo_article_urls(sitemap_index_url, start_year, max_articles_to_scan, allowed_sections):
    index_root = get_xml_root(sitemap_index_url)
    sitemap_entries = index_root.findall("ns:sitemap", namespace)

    kept_sitemaps = []

    for sitemap_elem in sitemap_entries:
        sitemap_url = safe_find_text(sitemap_elem, "ns:loc", namespace)
        sitemap_lastmod = safe_find_text(sitemap_elem, "ns:lastmod", namespace)

        if should_keep_sitemap(sitemap_url, sitemap_lastmod, start_year):
            kept_sitemaps.append(
                {
                    "sitemap_url": sitemap_url,
                    "sitemap_lastmod": sitemap_lastmod
                }
            )

    print(f"Total sitemap entries in index: {len(sitemap_entries)}")
    print(f"Sitemaps kept for processing: {len(kept_sitemaps)}")

    articles_data = []

    for sitemap_idx, sitemap_info in enumerate(kept_sitemaps, start=1):
        sitemap_url = sitemap_info["sitemap_url"]

        try:
            sitemap_root = get_xml_root(sitemap_url)
            url_entries = sitemap_root.findall("ns:url", namespace)

            for url_elem in url_entries:
                article_url = safe_find_text(url_elem, "ns:loc", namespace)
                lastmod_text = safe_find_text(url_elem, "ns:lastmod", namespace)

                if not is_probable_article_url(article_url):
                    continue

                article_dt = parse_iso_datetime(lastmod_text)

                if article_dt is not None:
                    if article_dt.year < start_year:
                        continue
                    year_month = f"{article_dt.year}-{article_dt.month:02d}"
                else:
                    parsed = urlparse(article_url)
                    path_parts = [part for part in parsed.path.split("/") if part]
                    year = int(path_parts[1])
                    month = int(path_parts[2])

                    if year < start_year:
                        continue

                    year_month = f"{year}-{month:02d}"

                section = guess_section_from_url(article_url)

                if section is None:
                    continue

                if section not in allowed_sections:
                    continue

                articles_data.append(
                    {
                        "article_url": article_url,
                        "year_month": year_month,
                        "section": section,
                        "lastmod": lastmod_text,
                        "source_sitemap": sitemap_url
                    }
                )

                if len(articles_data) >= max_articles_to_scan:
                    break

            if len(articles_data) >= max_articles_to_scan:
                break

        except Exception as e:
            print(f"Error processing sitemap {sitemap_url}: {e}")

        if sitemap_idx % 10 == 0:
            print(
                f"Processed sitemaps: {sitemap_idx}/{len(kept_sitemaps)} | "
                f"collected filtered articles: {len(articles_data)}"
            )

        time.sleep(sleep_between_requests)

    articles_df = pd.DataFrame(articles_data)
    articles_df = articles_df.drop_duplicates(subset=["article_url"]).reset_index(drop=True)

    print(f"Collected {len(articles_df)} filtered article URLs.")
    print(f"Sections used: {sorted(articles_df['section'].dropna().unique().tolist())}")
    return articles_df


def extract_origo_article_data(article_url, year_month, section, lastmod, source_sitemap):
    try:
        response = session.get(article_url, timeout=request_timeout_html)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        headline_tag = soup.find("h1")
        headline = clean_text(headline_tag.get_text(" ", strip=True)) if headline_tag else None

        image_url, alt_text, caption_text = extract_first_image_candidate(soup, article_url)

        if not image_url:
            return None

        return {
            "article_url": article_url,
            "headline": headline,
            "year_month": year_month,
            "section": section,
            "lastmod": lastmod,
            "source_sitemap": source_sitemap,
            "image_url": image_url,
            "caption": caption_text,
            "alt_text": alt_text,
            "image_index_in_article": 1
        }

    except Exception as e:
        print(f"Error processing article {article_url}: {e}")
        return None


#main pipeline for origo
articles_df = collect_origo_article_urls(
    sitemap_index_url=sitemap_index_url,
    start_year=start_year,
    max_articles_to_scan=max_articles_to_scan,
    allowed_sections=allowed_sections
)

saved_rows = []
seen_image_urls = set()
scanned_articles = 0

for _, row in articles_df.iterrows():
    if len(saved_rows) >= target_image_count:
        break

    scanned_articles += 1
    print(
        f"Scanning article {scanned_articles}/{len(articles_df)} | "
        f"saved {len(saved_rows)}/{target_image_count}"
    )

    article_data = extract_origo_article_data(
        article_url=row["article_url"],
        year_month=row["year_month"],
        section=row["section"],
        lastmod=row["lastmod"],
        source_sitemap=row["source_sitemap"]
    )

    if article_data is None:
        time.sleep(sleep_between_requests)
        continue

    image_url = article_data["image_url"]

    if not image_url or image_url in seen_image_urls:
        time.sleep(sleep_between_requests)
        continue

    filename = build_image_filename(
        image_url=image_url,
        year_month=article_data["year_month"],
        section=article_data["section"],
        image_index=len(saved_rows) + 1
    )

    save_path = os.path.join(images_folder, filename)

    local_path, _, _ = download_image(
        image_url=image_url,
        save_path=save_path,
        min_size=min_size
    )

    if local_path is None:
        time.sleep(sleep_between_requests)
        continue

    seen_image_urls.add(image_url)

    saved_rows.append(
        {
            "image_id": len(saved_rows) + 1,
            "article_url": article_data["article_url"],
            "headline": article_data["headline"],
            "year_month": article_data["year_month"],
            "section": article_data["section"],
            "lastmod": article_data["lastmod"],
            "source_sitemap": article_data["source_sitemap"],
            "image_url": article_data["image_url"],
            "caption": article_data["caption"],
            "alt_text": article_data["alt_text"],
            "image_index_in_article": article_data["image_index_in_article"],
            "local_image_path": local_path
        }
    )

    print(f"Saved {len(saved_rows)}/{target_image_count}: {local_path}")

    time.sleep(sleep_between_requests)

images_df = pd.DataFrame(saved_rows)
images_df.to_csv(metadata_csv_path, index=False, encoding="utf-8-sig")

print()
print(f"Finished. Saved {len(images_df)} images.")
print(f"Images folder: {images_folder}")
print(f"Metadata CSV: {metadata_csv_path}")
print()
print(images_df.head())

In [ ]:
#regex pattern based on stems for caption search
search_stems = [
    "ukrán",
    "ukrajn",
    "orosz",
    "oroszország",
    "donbasz",
    "donyeck",
    "luhanszk",
    "krím",
    "kijev",
    "moszkva",
    "kreml",
    "putyin",
    "zelenszk",
    "wagner",
    "prigozsin",
    "lavrov",
    "medvegyev",
    "háború",
    "inváz",
    "támad",
    "ostrom",
    "offenzív",
    "megszáll",
    "szankció",
    "fegyverszállít",
    "békétárgyal",
    "front"
]
images_df2["caption_clean"] = images_df2["caption"].fillna("").str.lower()

def find_matching_stems(text, stems):
    matches = []
    for stem in stems:
        if stem in text:
            matches.append(stem)
    return sorted(set(matches))

images_df2["caption_matched_keywords"] = images_df2["caption_clean"].apply(
    lambda text: find_matching_stems(text, search_stems)
)

images_df2["caption_match"] = images_df2["caption_matched_keywords"].apply(
    lambda x: len(x) > 0
)

caption_related_df2 = images_df2[images_df2["caption_match"]].copy()

ukraine_related_folder = os.path.join(output_base_folder, "ukraine_related")
os.makedirs(ukraine_related_folder, exist_ok=True)

copied2 = 0
missing2 = 0

for _, row in caption_related_df2.iterrows():
    source_path = row["local_image_path"]

    if not isinstance(source_path, str) or not os.path.exists(source_path):
        missing2 += 1
        continue

    filename = os.path.basename(source_path)
    target_path = os.path.join(ukraine_related_folder, filename)

    if not os.path.exists(target_path):
        shutil.copy2(source_path, target_path)
        copied2 += 1

caption_related_df2.to_csv(
    os.path.join(output_base_folder, "ukraine_related_metadata.csv"),
    index=False,
    encoding="utf-8-sig"
)

print("Total rows:", len(images_df2))
print("Rows with caption match:", len(caption_related_df2))
print(f"Copied images: {copied2}")
print(f"Missing files: {missing2}")
print(f"Folder: {ukraine_related_folder}")

In [ ]:
caption_related_df2[
    ["image_id", "caption", "caption_matched_keywords"]
].head(30)

In [ ]:
ukraine_related_folder = os.path.join(output_base_folder, "ukraine_related")
os.makedirs(ukraine_related_folder, exist_ok=True)

copied = 0
missing = 0

for _, row in caption_related_df2.iterrows():
    source_path = row["local_image_path"]

    if not isinstance(source_path, str) or not os.path.exists(source_path):
        missing += 1
        continue

    filename = os.path.basename(source_path)
    target_path = os.path.join(ukraine_related_folder, filename)

    if not os.path.exists(target_path):
        shutil.copy2(source_path, target_path)
        copied += 1

caption_related_df2.to_csv(
    os.path.join(output_base_folder, "ukraine_related_metadata.csv"),
    index=False,
    encoding="utf-8-sig"
)

print(f"Copied images: {copied}")
print(f"Missing files: {missing}")
print(f"Folder: {ukraine_related_folder}")

In [ ]:
pd.set_option("display.max_colwidth", None)

caption_related_df2[
    ["image_id", "headline", "caption", "caption_matched_keywords", "section", "year_month"]
].head(50)

## Full download

In [ ]:
import os

print("Working directory:")
print(os.getcwd())

print("\nFull image folder path:")
print(os.path.abspath("spiegel_images"))

In [ ]:
# main settings
sitemap_url = "https://www.spiegel.de/sitemaps/sitemap-articles.xml"
allowed_sections = ["politik", "ausland", "wirtschaft"]
years = ["2022", "2023", "2024", "2025", "2026"]
#limit = 100  # comment this line later to remove limit

# download sitemap
response = requests.get(sitemap_url)
soup = BeautifulSoup(response.content, "xml")
all_urls = [loc.text for loc in soup.find_all("loc")]

# filter by year directly in url
filtered_urls = [u for u in all_urls if any(f"/{y}/" in u for y in years)]

print("URLs after year filtering:", len(filtered_urls))

records = []
counter = 0

for url in tqdm(filtered_urls):

    if counter >= limit:
        break

    try:
        r = requests.get(url, timeout=10)
        article_soup = BeautifulSoup(r.content, "html.parser")

        section_tag = article_soup.find("meta", property="og:article:section")
        if not section_tag:
            continue

        section = section_tag.get("content")

        if section not in allowed_sections:
            continue

        date_tag = article_soup.find("meta", property="article:published_time")
        full_datetime = date_tag.get("content") if date_tag else None

        title_tag = article_soup.find("meta", property="og:title")
        title = title_tag.get("content") if title_tag else None

        text_paragraphs = article_soup.find_all("p")
        article_text = " ".join([p.get_text() for p in text_paragraphs])

        image_tag = article_soup.find("meta", property="og:image")
        image_url = image_tag.get("content") if image_tag else None

        records.append({
            "article_url": url,
            "section": section,
            "datetime_full": full_datetime,
            "title": title,
            "image_url": image_url,
            "text": article_text
        })

        counter += 1

    except:
        continue

articles_df = pd.DataFrame(records)

print("Collected articles:", len(articles_df))

In [ ]:
# create image folder
image_folder = "spiegel_images"
os.makedirs(image_folder, exist_ok=True)

for idx, row in tqdm(articles_df.iterrows(), total=len(articles_df)):

    image_url = row["image_url"]

    if not image_url:
        continue

    try:
        img_response = requests.get(image_url, timeout=10)
        file_path = os.path.join(image_folder, f"{idx}.jpg")

        with open(file_path, "wb") as f:
            f.write(img_response.content)

    except:
        continue